# RAG — Retrieval-Augmented Generation: Chunking, Embeddings y Vectorstores

> **Objetivo de este notebook:** Entender cómo construir un sistema RAG desde cero: cargar documentos, dividirlos en fragmentos (chunks), convertirlos en vectores (embeddings), almacenarlos en una base de datos vectorial (Chroma) y recuperar los más relevantes ante una pregunta.

---

## ¿Qué es RAG y por qué lo necesitamos?

Un LLM estándar tiene dos limitaciones críticas:
1. **No conoce tus datos privados** — solo sabe lo que vio en el entrenamiento
2. **Tiene fecha de corte** — no conoce información posterior a su entrenamiento

**RAG resuelve esto** dándole al LLM acceso a tus documentos en tiempo real:

```
FLUJO RAG:

Pregunta del usuario
        ↓
[RETRIEVER] Busca los fragmentos más relevantes en tu base de datos
        ↓
[LLM] Responde usando esos fragmentos como contexto
        ↓
Respuesta fundamentada en tus documentos
```

### Componentes de un sistema RAG:
| Componente | Qué hace | Herramienta en este notebook |
|-----------|----------|-----------------------------|
| **Document Loader** | Carga el PDF y extrae el texto | `PyPDFLoader` |
| **Text Splitter** | Divide el texto en chunks | `RecursiveCharacterTextSplitter`, etc. |
| **Embedding Model** | Convierte texto en vectores numéricos | `HuggingFaceEmbeddings` |
| **Vectorstore** | Almacena y busca vectores eficientemente | `Chroma` |
| **LLM** | Genera la respuesta final | `Gemini via LangChain` |

---

## PARTE 1 — Pipeline RAG Básico

Primero construimos un pipeline RAG completo y funcional con un contrato laboral como documento de prueba. Luego exploraremos las diferentes estrategias de chunking en la Parte 2.

### Instalación de dependencias

- `langchain-google-genai`: conector LangChain para Gemini
- `langchain-core`: núcleo de LangChain (prompts, parsers, etc.)
- `langchain-community`: loaders y utilidades de la comunidad
- `pypdf`: biblioteca para leer archivos PDF

In [ ]:
# Instalamos todo en una línea con -q (quiet) para no saturar la pantalla
# Ejecuta esta celda solo la primera vez en cada entorno
# !pip install langchain-google-genai langchain-core langchain-community pypdf

### Paso 1: Cargar el documento PDF

`PyPDFLoader` extrae el texto de cada página de un PDF y lo convierte en objetos `Document` de LangChain.

Cada `Document` tiene:
- `page_content` → el texto de la página
- `metadata` → información sobre el origen (nombre de archivo, número de página, etc.)

El metadata es importante para el RAG: cuando recuperas un chunk, sabes exactamente de qué página viene.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# Ruta al PDF (ajusta si tu archivo está en otra ubicación)
ruta = "./Montaje_de_un_Motor_Eléctrico_Básico.pdf"

# PyPDFLoader: cada página del PDF se convierte en un Document separado
loader = PyPDFLoader(ruta)

In [ ]:
# loader.load() lee todas las páginas y devuelve una lista de Document
# Una lista de Document = una lista de objetos, cada uno con texto + metadatos
pages = loader.load()

# Inspeccionamos el resultado
print(f"Número de páginas cargadas: {len(pages)}")
print(f"\nTipo del primer elemento: {type(pages[0])}")
print(f"\nMetadatos del primer Document: {pages[0].metadata}")
print(f"\nPrimeros 200 caracteres de la página 1:")
print(pages[0].page_content[:200])

In [ ]:
# Iteramos por las páginas para ver el contenido completo
# Observa el formato: cada página tiene su propio texto
for i, page in enumerate(pages):
    print(f"---- Página {i+1} -----")
    print(f"Contenido: {page.page_content[:300]}")
    print()

### Paso 2: Dividir el texto en chunks (fragmentos)

**¿Por qué dividir el texto?**

Los LLMs tienen un límite de tokens que pueden procesar a la vez (contexto máximo). Además, si envías un documento entero como contexto, el modelo se pierde entre información irrelevante.

La solución: **dividir el documento en fragmentos pequeños** y solo enviar los más relevantes a la pregunta.

**Parámetros clave de `RecursiveCharacterTextSplitter`:**
- `chunk_size`: tamaño máximo de cada fragmento (en caracteres)
- `chunk_overlap`: cuántos caracteres se solapan entre chunks consecutivos

**¿Por qué el overlap?** Para no perder contexto en los límites entre chunks. Si una frase importante queda partida entre dos chunks, el overlap garantiza que ambos chunks la tengan completa.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configuramos el splitter:
# chunk_size=1000 → cada fragmento tendrá máximo 1000 caracteres
# chunk_overlap=100 → 100 caracteres se repiten entre chunks adyacentes
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

In [ ]:
# Recargamos las páginas (nos aseguramos de tener la variable limpia)
pages = loader.load()

# split_documents() divide cada Document en chunks y preserva los metadatos originales
# Diferencia con split_text(): split_documents trabaja con objetos Document (mantiene metadata)
chunks = text_splitter.split_documents(pages)

print(f"Páginas originales: {len(pages)}")
print(f"Chunks generados: {len(chunks)}")
print(f"\nChunk de ejemplo (chunk 1):")
print(f"Texto: {chunks[0].page_content[:300]}")
print(f"\nMetadata del chunk: {chunks[0].metadata}")

### Paso 3: Configurar el LLM (Gemini)

Configuramos el modelo de lenguaje que usaremos para generar respuestas. En un sistema RAG, el LLM es el **generador**: recibe los chunks relevantes como contexto y produce la respuesta final.

> En Google Colab, las claves de API se guardan en **Secrets** (icono de llave en el panel izquierdo) y se acceden con `userdata.get()`.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata

MODEL = "gemini-2.5-flash-lite"         # Modelo rápido y económico de Google
API_KEY = userdata.get('GEMINI_API_KEY') # Recupera la clave guardada en Secrets de Colab

print("✅ API Key cargada")

In [ ]:
# Creamos el objeto LLM de LangChain
# temperature=0.5 → equilibrio entre precisión y creatividad
# Para RAG se suele usar temperature baja (0.0-0.3) para respuestas más factuales
llm = ChatGoogleGenerativeAI(
    model=MODEL,
    temperature=0.5,
    google_api_key=API_KEY
)

print(f"✅ Modelo {MODEL} listo")

### (Opcional) Resumir chunks con el LLM

El código comentado muestra cómo usar el LLM para resumir cada chunk. Está comentado porque hace una llamada a la API por cada chunk, lo que puede ser costoso si hay muchos.

En RAG real, no resumimos los chunks — los indexamos tal cual y dejamos que la búsqueda vectorial seleccione los más relevantes.

In [ ]:
# CÓDIGO OPCIONAL — Descomentar para probar
# Genera un resumen de los primeros 2 chunks usando el LLM

# resumenes = []
# for chunk in chunks[:2]:  # Solo los 2 primeros para no gastar demasiados tokens
#     response = llm.invoke(f"Haz un resumen de los puntos mas importantes del chunk: {chunk.page_content}")
#     resumenes.append(response.content)
#     print(response.content)
#     print("-" * 50)

---

## PARTE 2 — Pipeline RAG Completo con Contrato Laboral

Ahora construimos el pipeline RAG completo: embeddings + vectorstore + retrieval + LLM.

Usaremos un contrato laboral como documento para poder hacerle preguntas específicas sobre su contenido.

### Embeddings — Convertir texto en vectores

**¿Qué son los embeddings?**

Un embedding es la representación numérica de un texto como un vector de números reales. Textos con significado similar producen vectores similares (cercanos en el espacio vectorial).

```
"El contrato tiene una duración de 1 año" → [0.23, -0.41, 0.87, ..., 0.12]  (384 números)
"La duración del acuerdo es 12 meses"    → [0.21, -0.39, 0.85, ..., 0.14]  (vectores similares)
"El cielo es azul"                        → [-0.54, 0.73, -0.31, ..., 0.67]  (vector muy diferente)
```

**¿Por qué HuggingFace y no Gemini Embeddings?**

- `all-MiniLM-L6-v2` es un modelo gratuito que se ejecuta localmente (sin coste de API)
- Excelente equilibrio entre calidad y velocidad
- 384 dimensiones por vector
- Para producción con datos críticos, se prefieren embeddings de OpenAI o Google (más potentes pero de pago)

In [ ]:
# Instalar la integración de HuggingFace con LangChain
!pip install langchain-huggingface

In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

# La primera vez que ejecutes esto, descargará el modelo (~80MB)
# Las siguientes veces lo carga desde caché → mucho más rápido
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
    # Este modelo genera vectores de 384 dimensiones por chunk
    # Es uno de los modelos de embeddings más usados en la industria
)

print("✅ Modelo de embeddings cargado")

### Vectorstore Chroma — La base de datos vectorial

**¿Qué es un vectorstore?**

Es una base de datos especializada en almacenar y buscar vectores de forma eficiente. A diferencia de una base de datos SQL que busca por igualdad exacta, un vectorstore busca por **similitud semántica**.

**Flujo de indexado** (se hace una vez):
```
Chunks de texto
    → Embedding model → Vectores
    → Vectorstore → Almacena vectores + texto original
```

**Flujo de consulta** (se hace cada vez que hay una pregunta):
```
Pregunta del usuario
    → Embedding model → Vector de la pregunta
    → Vectorstore → Busca los k vectores más cercanos
    → Devuelve los chunks originales de esos vectores
```

**Chroma** es una base de datos vectorial open-source, fácil de usar y perfecta para desarrollo y proyectos de tamaño medio. Para producción a gran escala se usan Pinecone, Weaviate, Qdrant, etc.

In [ ]:
# !pip install langchain-chroma

In [ ]:
from langchain_chroma import Chroma

In [ ]:
# Cargamos el contrato laboral (documento con información específica)
ruta = "./Contrato_Laboral_Individual_I.pdf"
loader = PyPDFLoader(ruta)
pages = loader.load()

print(f"Páginas cargadas: {len(pages)}")

In [ ]:
# Dividimos el contrato en chunks más pequeños
# chunk_size=500 y chunk_overlap=50 son valores más pequeños que antes
# → Chunks más granulares = retrieval más preciso para preguntas concretas
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,   # Más pequeño que antes: el contrato tiene cláusulas concretas
    chunk_overlap=50  # 10% de overlap para no perder contexto en los cortes
)

docs_split = text_splitter.split_documents(pages)

print(f"Se han creado: {len(docs_split)} chunks de texto")
print(f"\nEjemplo de chunk:")
print(docs_split[0].page_content[:300])

In [ ]:
# Creamos el vectorstore:
# 1. Toma cada chunk de docs_split
# 2. Lo convierte en un vector usando el modelo de embeddings
# 3. Almacena los vectores + el texto original en Chroma
# persist_directory → guarda la base de datos en disco para no recrarlo cada vez
vectorstore = Chroma.from_documents(
    docs_split,
    embedding=embeddings,
    persist_directory="../data/chroma_db"  # Ruta donde se guarda la BD
)

print(f"✅ Vectorstore creado con {len(docs_split)} chunks indexados")

### Retrieval — Búsqueda semántica

`similarity_search()` convierte la consulta en un vector y busca los **k** chunks más cercanos en el vectorstore.

**"Más cercanos"** significa que el modelo de embeddings considera que esos chunks son semánticamente similares a la pregunta, aunque no usen las mismas palabras exactas.

In [ ]:
# La consulta en lenguaje natural
consulta = "Cual es el bonus de permanencia de CARLOS EDUARDO NAVARRO IBÁÑEZ"

# similarity_search:
# 1. Convierte la consulta en un vector
# 2. Busca los 4 chunks más cercanos en el espacio vectorial
# 3. Devuelve los Documents correspondientes
resultados = vectorstore.similarity_search(consulta, k=4)

print(f"Chunks recuperados: {len(resultados)}")

In [ ]:
# Inspeccionamos qué chunks encontró el retriever
# Estos serán el contexto que le pasaremos al LLM
print("Top documentos más similares a la consulta:")
for i, doc in enumerate(resultados, start=1):
    print(f"\n--- Resultado {i} ---")
    print(f"📄 Contenido: {doc.page_content}")
    print(f"📌 Metadatos: {doc.metadata}")

### Generación — LLM con contexto vs. sin contexto

Aquí está el núcleo de RAG: comparar la calidad de la respuesta **con** y **sin** contexto recuperado.

In [ ]:
# CON CONTEXTO (RAG): le pasamos los chunks recuperados como contexto
# El LLM puede leer los fragmentos del contrato para responder con datos reales
print("=== RESPUESTA CON RAG (con contexto) ===")
response_con_rag = llm.invoke(
    f"Cual es el bonus de permanencia de CARLOS EDUARDO NAVARRO IBÁÑEZ: {resultados}"
    # Aquí 'resultados' son los chunks del contrato recuperados por el vectorstore
)
print(response_con_rag.content)

In [ ]:
# SIN CONTEXTO (LLM puro): le preguntamos sin darle información del contrato
# El modelo no tiene ni idea de quién es Carlos Eduardo Navarro → puede alucinar
print("=== RESPUESTA SIN RAG (sin contexto) ===")
response_sin_rag = llm.invoke(
    "Cual es el bonus de permanencia de CARLOS EDUARDO NAVARRO IBÁÑEZ"
    # El LLM no tiene acceso al contrato → respuesta genérica o alucinación
)
print(response_sin_rag.content)

# Observa la diferencia: con RAG, el LLM cita datos reales del contrato.
# Sin RAG, el LLM inventa o admite que no sabe.

---

## PARTE 3 — Estrategias de Chunking: ¿Cómo dividir bien el texto?

El chunking es **la decisión más importante** en un sistema RAG. Un mal chunking provoca que el retriever recupere fragmentos irrelevantes o incompletos, lo que degrada la respuesta del LLM.

Veremos **4 estrategias** con sus ventajas y limitaciones:

| Estrategia | Criterio de corte | Cuándo usarla |
|-----------|-----------------|---------------|
| `CharacterTextSplitter` | Un separador fijo (ej: `\n\n`) | Texto muy estructurado, prototipado rápido |
| `RecursiveCharacterTextSplitter` | Jerarquía de separadores | Caso general — la más usada en producción |
| `SemanticChunker` | Cambios de significado (embeddings) | Texto denso con múltiples temas |
| `MarkdownHeaderTextSplitter` | Cabeceras Markdown | Documentación técnica en formato Markdown |

Usaremos el **paper original de RAG** (arxiv 2005.11401) como documento de prueba.

In [ ]:
import urllib.request

# Descargamos el paper original de RAG desde arxiv
# Este paper es el que propuso RAG como arquitectura (Lewis et al., 2020)
url = "https://arxiv.org/pdf/2005.11401"
urllib.request.urlretrieve(url, "rag_paper.pdf")
print("✅ Paper descargado como rag_paper.pdf")

In [ ]:
loader = PyPDFLoader("rag_paper.pdf")
pages = loader.load()

print(f"Páginas cargadas: {len(pages)}")
print(f"Caracteres en página 1: {len(pages[0].page_content)}")
print(f"\n-- Muestra de página 1 --\n{pages[0].page_content[:500]}")

In [ ]:
# Necesitamos langchain-classic para el objeto Document
# (versión legacy que se usa en algunos ejemplos)
!pip install langchain-classic

In [ ]:
from langchain_classic.schema import Document

# Concatenamos todas las páginas en un único Document
# Razón: queremos que los splitters puedan cortar en cualquier punto del texto,
# no solo dentro de cada página individualmente
full_text = "\n\n".join([p.page_content for p in pages])
full_doc = [Document(
    page_content=full_text,
    metadata={"source": "rag_paper.pdf"}
)]

print(f"Total caracteres en el documento: {len(full_text):,}")
print(f"Total palabras aproximadas: {len(full_text.split()):,}")

### Estrategia 1: CharacterTextSplitter — El más simple

**Cómo funciona:**
Busca un único separador (en este caso `\n\n`) y corta el texto cada vez que lo encuentra. Si el chunk resultante supera `chunk_size`, lo corta por caracteres.

**Problema:** Es rígido. Si el texto no tiene `\n\n` frecuentes, puede generar chunks muy grandes o muy pequeños.

**Cuándo usarlo:** Texto muy estructurado con separadores consistentes (logs, CSVs, texto con formato fijo).

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

splitter_char = CharacterTextSplitter(
    separator="\n\n",   # Corta SOLO en párrafos (doble salto de línea)
    chunk_size=800,     # Tamaño máximo del chunk en caracteres
    chunk_overlap=100,  # 100 caracteres repetidos entre chunks consecutivos
    length_function=len # La función que mide el tamaño del texto (por defecto: len)
)

In [ ]:
chunks_char = splitter_char.split_documents(full_doc)

# Calculamos estadísticas básicas sobre los chunks generados
tamaños = [len(c.page_content) for c in chunks_char]
print(f"Chunks generados: {len(chunks_char)}")
print(f"Tamaño medio: {sum(tamaños)/len(tamaños):.0f} caracteres")
print(f"Tamaño mínimo: {min(tamaños)} | Máximo: {max(tamaños)}")
print(f"\nChunk de ejemplo (índice 2):")
print(chunks_char[2].page_content)

### Estrategia 2: RecursiveCharacterTextSplitter — El más usado en producción

**Cómo funciona:**
Prueba una **jerarquía de separadores** en orden hasta encontrar el que produce chunks del tamaño correcto:

```
1. Intenta cortar por \n\n (párrafos)
2. Si el chunk sigue siendo muy grande, prueba \n (líneas)
3. Si sigue siendo grande, prueba ". " (oraciones)
4. Si sigue grande, prueba " " (palabras)
5. Como último recurso, corta en cualquier carácter
```

**Ventaja:** Produce chunks más naturales y de tamaño más uniforme. Raramente corta en mitad de una oración.

**Es la opción por defecto recomendada para la mayoría de casos.**

In [ ]:
# RecursiveCharacterTextSplitter con jerarquía de separadores explícita
splitter_recursive = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    length_function=len,
    # Orden de prioridad de separadores:
    # Primero intenta párrafos, luego líneas, luego frases, luego palabras, luego caracteres
    separators=["\n\n", "\n", ". ", " ", ""]
)

In [ ]:
chunks_recursiva = splitter_recursive.split_documents(full_doc)

tamaños = [len(c.page_content) for c in chunks_recursiva]
print(f"Chunks generados: {len(chunks_recursiva)}")
print(f"Tamaño medio: {sum(tamaños)/len(tamaños):.0f} caracteres")
print(f"Tamaño mínimo: {min(tamaños)} | Máximo: {max(tamaños)}")
print(f"\nChunk de ejemplo (índice 2):")
print(chunks_recursiva[2].page_content)

# Compara la distribución de tamaños con CharacterTextSplitter
# RecursiveCharacter debería tener una distribución más uniforme

### Estrategia 3: SemanticChunker — Corte por significado

**Cómo funciona:**
En lugar de cortar por caracteres, usa embeddings para detectar dónde **cambia el tema** del texto.

1. Divide el texto en frases
2. Calcula el embedding de cada frase
3. Detecta dónde la similitud entre frases consecutivas cae por debajo de un umbral → ahí corta

**Parámetro `breakpoint_threshold_type`:**
- `"percentile"`: el umbral de corte es el percentil N de las diferencias de similitud
- `breakpoint_threshold_amount=95`: solo corta donde la diferencia de significado está en el 5% más alto

**Ventaja:** Chunks semánticamente coherentes — cada chunk habla de un único tema.

**Desventaja:** Más lento (necesita calcular embeddings para cada frase) y los chunks pueden tener tamaños muy variables.

In [ ]:
!pip install langchain_experimental

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker

# SemanticChunker usa el modelo de embeddings para detectar cambios de tema
# embeddings: el mismo modelo de HuggingFace que ya cargamos
# breakpoint_threshold_type="percentile": usa percentiles para detectar rupturas semánticas
# breakpoint_threshold_amount=95: solo corta en los cambios de tema más bruscos (top 5%)
splitter_semantic = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=95
    # Valor más bajo → más cortes (chunks más pequeños y más temáticos)
    # Valor más alto → menos cortes (chunks más grandes, menos precisos)
)

In [ ]:
# Atención: esta celda puede tardar varios minutos
# Está calculando embeddings para cada frase del paper para detectar cambios semánticos
print("Calculando embeddings frase a frase para detectar cambios semánticos...")
chunk_semantic = splitter_semantic.split_documents(full_doc)

tamaños = [len(c.page_content) for c in chunk_semantic]
print(f"\nChunks generados: {len(chunk_semantic)}")
print(f"Tamaño medio: {sum(tamaños)/len(tamaños):.0f} caracteres")
print(f"Tamaño mínimo: {min(tamaños)} | Máximo: {max(tamaños)}")

# Observa que los tamaños son MUY variables — el SemanticChunker no respeta chunk_size
# Cada chunk tiene el tamaño que "necesita" para mantener coherencia semántica

In [ ]:
# Ver el primer chunk semántico — debería ser un bloque temáticamente coherente
print("Chunk semántico 0:")
print(chunk_semantic[0].page_content[:500])

### Estrategia 4: MarkdownHeaderTextSplitter — Corte por estructura

**Cómo funciona:**
Corta el texto usando las **cabeceras Markdown** (`#`, `##`, `###`) como puntos de división y añade esas cabeceras a los metadatos de cada chunk.

**Ventaja clave:** Los metadatos enriquecidos permiten al retriever saber exactamente en qué sección del documento está cada chunk, lo que mejora la trazabilidad de las respuestas.

**Cuándo usarlo:** Documentación técnica, wikis, README, cualquier texto bien estructurado con Markdown.

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

# Documento Markdown de ejemplo sobre RAG
# Fíjate en la estructura jerárquica de cabeceras
markdown_doc = """
# Introducción a RAG

RAG combina recuperación de documentos con generación de texto.
Es especialmente útil para preguntas sobre documentos específicos.

## Componentes principales

Los componentes de un sistema RAG son el retriever y el generador.
El retriever busca documentos relevantes en una base vectorial.

### El Retriever

El retriever convierte la query en un vector y busca los más cercanos.
Usa métricas como cosine similarity o producto escalar.

### El Generador

El generador (LLM) recibe el contexto recuperado y genera la respuesta.
Modelos como GPT-4 o Claude son opciones comunes.

## Evaluación

Evaluar RAG requiere métricas especializadas como RAGAS.
Las métricas principales son faithfulness y answer relevancy.
"""

# Definimos qué cabeceras usar como puntos de corte y qué nombre darles en metadata
headers_to_split_on = [
    ("#",   "Header 1"),  # Cabecera de nivel 1 → metadata["Header 1"]
    ("##",  "Header 2"),  # Cabecera de nivel 2 → metadata["Header 2"]
    ("###", "Header 3"),  # Cabecera de nivel 3 → metadata["Header 3"]
]

In [ ]:
splitter_md = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

# split_text (no split_documents) porque el input es un string, no un Document
chunks_md = splitter_md.split_text(markdown_doc)

print(f"Chunks generados: {len(chunks_md)}")
print()

for i, chunk in enumerate(chunks_md):
    print(f"--- Chunk {i+1} ---")
    # El metadata ahora incluye las cabeceras de la sección
    # Esto permite saber en qué parte del documento está el chunk
    print(f"📌 Metadata (sección): {chunk.metadata}")
    print(f"📝 Contenido: {chunk.page_content[:200]}")
    print()

---

## PARTE 4 — Comparativa: ¿Qué estrategia produce mejor retrieval?

Creamos un vectorstore para cada estrategia y comparamos qué chunks recupera cada una ante las mismas preguntas.

**Conclusiones que verás:**
- `CharacterTextSplitter`: chunks grandes y variables → a veces recupera demasiado contexto irrelevante
- `RecursiveCharacterTextSplitter`: chunks más uniformes → retrieval más consistente
- `SemanticChunker`: chunks temáticamente puros → retrieval muy preciso pero puede ser muy amplio

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

# Calculamos estadísticas comparativas de las 3 estrategias
strategies = {
    "CharacterTextSplitter": chunks_char,
    "RecursiveCharacter\nTextSplitter": chunks_recursiva,
    "SemanticChunker": chunk_semantic,
}

stats = []
for name, chunks in strategies.items():
    sizes = [len(c.page_content) for c in chunks]
    stats.append({
        "Estrategia": name,
        "Nº chunks": len(chunks),
        "Tamaño medio": np.mean(sizes),
        "Tamaño mínimo": np.min(sizes),
        "Tamaño máximo": np.max(sizes),
        "Desviación estándar": np.std(sizes),
    })

df_stats = pd.DataFrame(stats).set_index("Estrategia")
print("Estadísticas comparativas de las estrategias de chunking:")
print(df_stats.round(0).to_string())

# Interpretación:
# - Mayor Nº chunks → fragmentación más granular
# - Menor desviación estándar → chunks más uniformes (mejor para retrieval predecible)

In [ ]:
import os
import shutil

def build_vectorstore(chunks, embeddings, persist_dir):
    """
    Crea un vectorstore Chroma desde una lista de chunks.
    Si ya existe en disco, lo elimina y lo recrea (evita inconsistencias).
    
    Args:
        chunks: lista de Documents con el texto dividido
        embeddings: modelo de embeddings a usar
        persist_dir: ruta donde se guarda el vectorstore en disco
    
    Returns:
        Chroma vectorstore listo para consultas
    """
    if os.path.exists(persist_dir):
        shutil.rmtree(persist_dir)  # Borra el directorio si ya existe
    return Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=persist_dir,
    )

# Indexamos los chunks de cada estrategia en su propio vectorstore
# Usamos HuggingFace (no Gemini) para el indexado → sin coste de API en esta fase
print("Indexando CharacterTextSplitter...")
db_char = build_vectorstore(chunks_char, embeddings, "./chroma_char")

print("Indexando RecursiveCharacterTextSplitter...")
db_recursive = build_vectorstore(chunks_recursiva, embeddings, "./chroma_recursive")

print("Indexando SemanticChunker...")
db_semantic = build_vectorstore(chunk_semantic, embeddings, "./chroma_semantic")

print("\n✅ Los 3 vectorstores están listos")

In [ ]:
def compare_retrieval(query, vectorstores_dict, k=3):
    """
    Compara los chunks recuperados por cada estrategia ante la misma query.
    
    Permite ver empíricamente cuál estrategia devuelve los fragmentos
    más relevantes y de qué tamaño son.
    
    Args:
        query: pregunta en lenguaje natural
        vectorstores_dict: diccionario {nombre: vectorstore}
        k: número de chunks a recuperar por estrategia
    """
    print(f"\n{'='*70}")
    print(f"🔍 QUERY: {query}")
    print(f"{'='*70}")

    for name, db in vectorstores_dict.items():
        # similarity_search devuelve los k chunks más similares a la query
        docs = db.similarity_search(query, k=k)
        print(f"\n📚 [{name}] — {len(docs)} chunks recuperados")
        print("-" * 50)
        for i, doc in enumerate(docs):
            # Mostramos tamaño del chunk y los primeros 180 caracteres
            print(f"  Chunk {i+1} ({len(doc.page_content)} chars): {doc.page_content[:180].strip()}...")


vectorstores = {
    "CharacterTextSplitter": db_char,
    "RecursiveCharacterTextSplitter": db_recursive,
    "SemanticChunker": db_semantic,
}

# Queries de prueba sobre el paper de RAG
# Observa cómo cada estrategia recupera fragmentos diferentes para la misma pregunta
queries = [
    "How does RAG combine retrieval with generation?",
    "What are the limitations of standard language models?",
    "How is the retriever trained in RAG?",
]

for query in queries:
    compare_retrieval(query, vectorstores, k=2)

---

## Resumen Final

### ¿Qué hemos aprendido?

**Pipeline RAG completo:**
```
PDF → PyPDFLoader → Documents
         ↓
    TextSplitter → Chunks
         ↓
   EmbeddingModel → Vectores
         ↓
      Chroma → Vectorstore (indexado)
         ↓
  Query del usuario → Vectores → similarity_search → k Chunks relevantes
         ↓
     LLM + contexto → Respuesta fundamentada
```

### Cuándo usar cada estrategia de chunking:

| Estrategia | Úsala cuando... | Evítala cuando... |
|-----------|----------------|-------------------|
| `CharacterTextSplitter` | El texto tiene separadores claros y consistentes | El texto es fluido sin estructura clara |
| `RecursiveCharacterTextSplitter` | Caso general (primera opción siempre) | Rara vez es mala opción |
| `SemanticChunker` | El texto mezcla muchos temas y quieres pureza semántica | Tienes un presupuesto ajustado (más lento y costoso) |
| `MarkdownHeaderTextSplitter` | El documento está en Markdown bien estructurado | El documento no tiene cabeceras Markdown |

### Regla práctica:
> **Empieza siempre con `RecursiveCharacterTextSplitter` con `chunk_size=500-1000` y `chunk_overlap=10-20%`. Solo cambia si los resultados son malos.**